In [6]:
"""
Scrape the first 100 books (pages 1-5) from books.toscrape.com
into books.csv with columns: title, price, rating, in_stock, url
"""
import csv
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
OUTPUT_FILE = "books.csv"
NUM_PAGES = 5

RATING_WORDS = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}


def parse_price(raw_price: str) -> float:
    """'£51.77' -> 51.77"""
    cleaned = raw_price.replace("£", "").replace("Â", "").strip()
    return float(cleaned)


def parse_rating(tag_classes) -> int:
    """['star-rating', 'Three'] -> 3"""
    for cls in tag_classes:
        if cls in RATING_WORDS:
            return RATING_WORDS[cls]
    return 0


def parse_stock(raw_text: str) -> bool:
    """'In stock (22 available)' -> True, anything else -> False"""
    return "in stock" in raw_text.strip().lower()


def scrape_page(page_num: int):
    url = BASE_URL.format(page_num)
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    books = []
    for article in soup.select("article.product_pod"):
        title = article.h3.a["title"].strip()
        relative_link = article.h3.a["href"]
        # Links on catalogue pages are relative to /catalogue/
        book_url = "https://books.toscrape.com/catalogue/" + relative_link.replace("../../../", "")

        price_text = article.select_one(".price_color").text
        price = parse_price(price_text)

        rating_tag = article.select_one("p.star-rating")
        rating = parse_rating(rating_tag["class"])

        stock_text = article.select_one(".instock.availability").text
        in_stock = parse_stock(stock_text)

        books.append(
            {
                "title": title,
                "price": price,
                "rating": rating,
                "in_stock": in_stock,
                "url": book_url,
            }
        )
    return books


def main():
    all_books = []
    for page in range(1, NUM_PAGES + 1):
        print(f"Scraping page {page}...")
        all_books.extend(scrape_page(page))
        time.sleep(0.5)  # be polite to the server

    all_books = all_books[:100]  # exactly the first 100 books

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "price", "rating", "in_stock", "url"])
        writer.writeheader()
        for book in all_books:
            row = dict(book)
            row["in_stock"] = "true" if row["in_stock"] else "false"
            writer.writerow(row)

    print(f"Wrote {len(all_books)} books to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Wrote 100 books to books.csv


In [7]:
%%writefile queries.sql
-- 1. Average price for each rating
SELECT rating, ROUND(AVG(price), 2) AS avg_price
FROM books
GROUP BY rating
ORDER BY rating;

-- 2. The 5 most expensive books rated 4 or 5
SELECT title, price, rating
FROM books
WHERE rating IN (4, 5)
ORDER BY price DESC
LIMIT 5;

-- 3. How many books are out of stock, per rating
SELECT rating, COUNT(*) AS out_of_stock_count
FROM books
WHERE in_stock = 0
GROUP BY rating
ORDER BY rating;

Overwriting queries.sql


In [8]:
import csv
import sqlite3

conn = sqlite3.connect("books.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS books")
cur.execute("""
    CREATE TABLE books (
        title    TEXT,
        price    REAL,
        rating   INTEGER,
        in_stock BOOLEAN,
        url      TEXT
    )
""")

with open("books.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = [
        (row["title"], float(row["price"]), int(row["rating"]),
         row["in_stock"].strip().lower() == "true", row["url"])
        for row in reader
    ]

cur.executemany(
    "INSERT INTO books (title, price, rating, in_stock, url) VALUES (?, ?, ?, ?, ?)",
    rows,
)
conn.commit()
print(f"Loaded {len(rows)} rows into books.db")

Loaded 100 rows into books.db


In [9]:
with open("queries.sql", encoding="utf-8") as f:
    sql_script = f.read()

queries = [q.strip() for q in sql_script.split(";") if q.strip()]

for i, query in enumerate(queries, start=1):
    print(f"\n--- Query {i} ---")
    print(query)
    print("-" * 40)
    cur.execute(query)
    cols = [desc[0] for desc in cur.description]
    print(" | ".join(cols))
    for row in cur.fetchall():
        print(" | ".join(str(v) for v in row))


--- Query 1 ---
-- 1. Average price for each rating
SELECT rating, ROUND(AVG(price), 2) AS avg_price
FROM books
GROUP BY rating
ORDER BY rating
----------------------------------------
rating | avg_price
1 | 35.52
2 | 35.91
3 | 36.84
4 | 33.98
5 | 30.01

--- Query 2 ---
-- 2. The 5 most expensive books rated 4 or 5
SELECT title, price, rating
FROM books
WHERE rating IN (4, 5)
ORDER BY price DESC
LIMIT 5
----------------------------------------
title | price | rating
The Death of Humanity: and the Case for Life | 58.11 | 4
The Past Never Ends | 56.5 | 4
Sapiens: A Brief History of Humankind | 54.23 | 5
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) | 52.29 | 5
Behind Closed Doors | 52.22 | 4

--- Query 3 ---
-- 3. How many books are out of stock, per rating
SELECT rating, COUNT(*) AS out_of_stock_count
FROM books
WHERE in_stock = 0
GROUP BY rating
ORDER BY rating
----------------------------------------
rating | out_of_stock_count


In [10]:
from google.colab import files
files.download("books.csv")
files.download("queries.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>